# 1. Install + Import Requirements

In [113]:
%pip install mujoco mediapy numpy

import mujoco
import mediapy as media
import numpy as np

%pip install ultralytics

from ultralytics import YOLO

%pip install opencv-python


[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


# 2. Define MuJoCo XML Scene

In [114]:
xml_string = """
<mujoco>
  <compiler angle="degree"/>
  
  <asset>
    <!-- Define the .obj files as mesh assets -->
    <mesh name="mug_mesh" file="mug.obj"/>
    <mesh name="can_opener_mesh" file="can_opener.obj"/>
    <mesh name="action_fig_mesh" file="action_fig.obj"/>
    <mesh name="shoe_mesh" file="shoe.obj"/>
  </asset>

  <worldbody>
    <light pos="0 0 1.5" dir="0 0 -1" directional="true" diffuse="0.8 0.8 0.8"/>
    <geom type="plane" size="1 1 0.1" rgba="0.9 0.9 0.9 1"/>
    
    <!-- Object: Mug -->
    <body name="mug" pos="0 0 0.1">
      <freejoint/>
      <geom type="mesh" mesh="mug_mesh" rgba="0.8 0.2 0.2 1"/>
    </body>
    
    <!-- Object: Can Opener -->
    <body name="can_opener" pos="0.1 0 0.1">
      <freejoint/>
      <geom type="mesh" mesh="can_opener_mesh" rgba="0.2 0.8 0.2 1"/>
    </body>
    
    <!-- Object: Action Figure -->
    <body name="action_fig" pos="-0.1 0 0.1">
      <freejoint/>
      <geom type="mesh" mesh="action_fig_mesh" rgba="0.2 0.2 0.8 1"/>
    </body>
    
    <!-- Object: Shoe -->
    <body name="shoe" pos="0 0.1 0.1">
      <freejoint/>
      <geom type="mesh" mesh="shoe_mesh" rgba="0.5 0.5 0.5 1"/>
    </body>
  </worldbody>
</mujoco>
"""
model = mujoco.MjModel.from_xml_string(xml_string)
data = mujoco.MjData(model)

# 3. Generate Training Data for YOLO Finetuning
This will be labelled bounding boxes from openCV of the objects initialized to random transforms

In [115]:
import os
import cv2
import yaml

skip_perception_training = os.path.exists("runs/obb/current")

dataset_dir = "datasets/mujoco_obb"

# all objects have class 0, because we dont care what they are - just need to sweep them.
classes = {"mug": 0, "can_opener": 0, "action_fig": 0, "shoe": 0}
object_names = list(classes.keys())         # must be after classes dict

# use MuJoCo's built in segmenter
seg_renderer = mujoco.Renderer(model, 480, 640)
seg_renderer.enable_segmentation_rendering()
renderer = mujoco.Renderer(model, 480, 640)  # must be after seg_renderer

if skip_perception_training:
    print("Skipping dataset generation (using runs/obb/current).")
else:

    # set up datasets
    os.makedirs(f"{dataset_dir}/images/train", exist_ok=True)
    os.makedirs(f"{dataset_dir}/labels/train", exist_ok=True)

    # generates 100 training images
    print("Generating synthetic data. This will take a moment...")
    frames_to_generate = 100
    vis_img = None

    for i in range(frames_to_generate):
        # random positions and rotations for each frame
        for obj_name in object_names:
            body_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, obj_name)
            
            rand_x, rand_y = np.random.uniform(-0.4, 0.4, size=2)

            # adding random rotation
            rand_angle = np.random.uniform(0, 2 * np.pi)
            q_z = [np.cos(rand_angle/2), 0, 0, np.sin(rand_angle/2)]
            
            qpos_start = model.jnt_qposadr[model.body_jntadr[body_id]]
            data.qpos[qpos_start:qpos_start+3] = [rand_x, rand_y, 0.2]
            data.qpos[qpos_start+3:qpos_start+7] = q_z
            
        mujoco.mj_forward(model, data)
        num_steps = np.random.randint(10, 100)  # let the objects fall a random amount, would like mid-air support
        for _ in range(num_steps):  # roll out physics
            mujoco.mj_step(model, data)
            
        # render scene and segments
        renderer.update_scene(data)
        rgb_img = renderer.render()
        
        seg_renderer.update_scene(data)
        seg_img = seg_renderer.render()
        
        img_h, img_w = rgb_img.shape[:2]
        
        # save the last frame to see if the model is working correctly
        if i == frames_to_generate - 1:
            vis_img = rgb_img.copy()

        # now we can make the label for this data point
        label_file = open(f"{dataset_dir}/labels/train/frame_{i}.txt", "w")
        
        for obj_name, class_idx in classes.items():
            body_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, obj_name)
            
            # get the MuJoCo geoms
            geom_start = model.body_geomadr[body_id]
            geom_num = model.body_geomnum[body_id]
            
            # create binary mask for this object
            mask = np.isin(seg_img[:, :, 0], range(geom_start, geom_start + geom_num)).astype(np.uint8) * 255
            
            # use openCV to get the bounding box
            contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            
            if contours:
                c = max(contours, key=cv2.contourArea)
                # this next part is to only save the bounding box if not obscured or offscreen
                if cv2.contourArea(c) > 50: 
                    # minAreaRect gets the box corners
                    rect = cv2.minAreaRect(c)
                    box = cv2.boxPoints(rect)
                    
                    if vis_img is not None:
                        # draw a green box in the visualization image for us to see if its working
                        cv2.drawContours(vis_img, [np.int32(box)], 0, (0, 255, 0), 2)
                    
                    # must normalize coordinates between 0.0 and 1.0 to work in YOLO
                    box[:, 0] /= img_w
                    box[:, 1] /= img_h
                    
                    # yolo format: "class_idx x1 y1 x2 y2 x3 y3 x4 y4"
                    label_str = " ".join([f"{coord:.5f}" for coordinate in box for coord in coordinate])
                    label_file.write(f"{class_idx} {label_str}\n")
                    
        label_file.close()
        
        # create image for this data point, opencv uses BGR format
        cv2.imwrite(f"{dataset_dir}/images/train/frame_{i}.jpg", cv2.cvtColor(rgb_img, cv2.COLOR_RGB2BGR))

    print("Synthetic Dataset Generation Complete!")

    # view the final frame with the visualization as a sanity check
    if vis_img is not None:
        print("Visual sanity check:")
        media.show_image(vis_img)

Skipping dataset generation (using runs/obb/current).


# 4. Fine Tune YOLO
Use the data set of labelled images we just generated.

In [116]:
if skip_perception_training:
    print("Skipping YOLO training (using runs/obb/current).")
else:
    # ultralytics needs dataset.yaml, so we'll have to make it
    yaml_config = {
        'path': os.path.abspath(dataset_dir),
        'train': 'images/train',
        'val': 'images/train',
        # only one class
        'names': {0: "object"}
    }

    with open('mujoco_dataset.yaml', 'w') as f:
        yaml.dump(yaml_config, f)

    # import base YOLO
    model_obb = YOLO('yolov8n-obb.pt')

    # finetuning the YOLO
    results = model_obb.train(data='mujoco_dataset.yaml', epochs=25, imgsz=640, device="cpu", verbose=False)

    print("Training finished!")

Skipping YOLO training (using runs/obb/current).


# 4. Test Perception Module

In [117]:
class PerceptionModule:
    """
    Combines YOLO 2D Object Detection with Depth maps to estimate
    the 3D transform (position and orientation) of detected objects
    relative to the camera.
    """
    def __init__(self, weights_path, fovy_degrees=45.0):
        self.model = YOLO(weights_path)
        self.fovy_degrees = fovy_degrees

    def _rotation_from_obb(self, rotation_rad):
        # Building a camera-frame rotation from the OBB angle in the image plane.
        c = float(np.cos(rotation_rad))
        s = float(np.sin(rotation_rad))
        x_axis = np.array([c, s, 0.0], dtype=np.float32)
        z_axis = np.array([0.0, 0.0, -1.0], dtype=np.float32)
        y_axis = np.cross(z_axis, x_axis)
        x_axis /= np.linalg.norm(x_axis) + 1e-8
        y_axis /= np.linalg.norm(y_axis) + 1e-8
        z_axis = np.cross(x_axis, y_axis)
        return np.stack([x_axis, y_axis, z_axis], axis=1)

    def get_transforms(self, rgb_image, depth_image, model=None, camera=None):
        # run model
        results = self.model(rgb_image, verbose=False)[0]

        objects = []
        if results.obb is not None:
            h, w = rgb_image.shape[:2]

            # get camera intrinsics
            fovy_rad = np.deg2rad(self.fovy_degrees)
            fy = (h / 2.0) / np.tan(fovy_rad / 2.0)
            fx = fy
            cx = w / 2.0
            cy = h / 2.0

            # Mirror camera-frame axes to match the overhead camera frame in the scene.
            mirror_mat = np.diag([1.0, -1.0, -1.0]).astype(np.float32)

            cam_pos = None
            cam_R = None
            if model is not None and camera is not None:
                cam_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_CAMERA, camera)
                if cam_id >= 0:
                    cam_pos = model.cam_pos[cam_id].copy()
                    cam_quat = model.cam_quat[cam_id].copy()
                    cam_mat = np.zeros(9, dtype=np.float64)
                    mujoco.mju_quat2Mat(cam_mat, cam_quat)
                    cam_R = cam_mat.reshape(3, 3)

            for box in results.obb:
                # get OBB parameters
                x_center, y_center, width, height, rotation = box.xywhr[0].cpu().numpy()
                class_id = int(box.cls[0].item())
                confidence = float(box.conf[0].item())

                # get depth at each object's center
                u = int(np.clip(x_center, 0, w - 1))
                v = int(np.clip(y_center, 0, h - 1))
                z = depth_image[v, u]

                # use deprojection formula:
                x = (u - cx) * z / fx
                y = (v - cy) * z / fy
                pos_cam = mirror_mat @ np.array([x, y, z], dtype=np.float32)

                rot_cam = self._rotation_from_obb(rotation)
                rot_cam = mirror_mat @ rot_cam
                tf_cam = np.eye(4, dtype=np.float32)
                tf_cam[:3, :3] = rot_cam
                tf_cam[:3, 3] = pos_cam

                transform = {
                    "class": results.names[class_id],
                    "position_camera_frame": pos_cam,
                    "rotation_rad": float(rotation),
                    "rotation_camera_frame": rot_cam,
                    "transform_camera_frame": tf_cam,
                    "confidence": confidence,
                    "pixel_u": u,
                    "pixel_v": v
                }
                if cam_pos is not None and cam_R is not None:
                    pos_world = cam_pos + cam_R @ pos_cam
                    rot_world = cam_R @ rot_cam
                    tf_world = np.eye(4, dtype=np.float32)
                    tf_world[:3, :3] = rot_world
                    tf_world[:3, 3] = pos_world
                    transform["position_world"] = pos_world
                    transform["rotation_world"] = rot_world
                    transform["transform_world"] = tf_world
                objects.append(transform)

        # returns the list of object transforms and the annotated image
        return objects, results.plot()

# 5. Scene with Robot (Broom + Franka Panda)
Build the combined MuJoCo scene using `mujoco.MjSpec`.

The broom mesh is generated **programmatically** â€” correct dimensions,
zero licensing issues, no external conversion needed.

In [118]:
import subprocess, os

# --- 0. Generate broom.obj (but only if not already present ---)
def _make_broom_obj(path, n_seg=20):
    import numpy as np
    verts, faces = [], []
    r, hz0, hz1 = 0.015, 0.05, 0.75
    for i in range(n_seg):
        a = 2*np.pi*i/n_seg; verts.append((r*np.cos(a), r*np.sin(a), hz0))
    for i in range(n_seg):
        a = 2*np.pi*i/n_seg; verts.append((r*np.cos(a), r*np.sin(a), hz1))
    verts += [(0,0,hz0),(0,0,hz1)]
    bc, tc = 2*n_seg+1, 2*n_seg+2
    for i in range(n_seg):
        v0,v1,v2,v3 = i+1, i%n_seg+2, n_seg+i%n_seg+2, n_seg+i+1
        faces += [(v0,v1,v2),(v0,v2,v3)]
    for i in range(n_seg): faces.append((bc, i%n_seg+2, i+1))
    for i in range(n_seg): faces.append((tc, n_seg+i+1, n_seg+i%n_seg+2))
    bx,by,bz0,bz1 = 0.12,0.03,0.0,0.05; o = len(verts)+1
    for bv in [(-bx,-by,bz0),(bx,-by,bz0),(bx,by,bz0),(-bx,by,bz0),
               (-bx,-by,bz1),(bx,-by,bz1),(bx,by,bz1),(-bx,by,bz1)]:
        verts.append(bv)
    faces += [(o,o+2,o+1),(o,o+3,o+2),(o+4,o+5,o+6),(o+4,o+6,o+7),
              (o,o+1,o+5),(o,o+5,o+4),(o+1,o+2,o+6),(o+1,o+6,o+5),
              (o+2,o+3,o+7),(o+2,o+7,o+6),(o+3,o,o+4),(o+3,o+4,o+7)]
    with open(path,"w") as f:
        f.write("# Parametric broom: handle r=0.015m z=[0.05,0.75]; head 0.24x0.06x0.05m\n")
        for v in verts: f.write(f"v {v[0]:.6f} {v[1]:.6f} {v[2]:.6f}\n")
        for fc in faces: f.write(f"f {fc[0]} {fc[1]} {fc[2]}\n")
    print(f"Generated {path}  ({len(verts)} verts, {len(faces)} faces)")

if not os.path.exists("broom.obj"):
    _make_broom_obj("broom.obj")
else:
    print("broom.obj already present")

# --- 1. Download Franka Panda model (mujoco_menagerie) ---
if not os.path.exists("mujoco_menagerie"):
    subprocess.run(
        ["git", "clone", "--depth", "1",
         "https://github.com/google-deepmind/mujoco_menagerie.git"],
        check=True,
    )

PANDA_XML = "mujoco_menagerie/franka_emika_panda/panda.xml"

# --- 2. Base scene XML (broom mesh included directly) ---
# Broom geometry (single geom = visual + collision):
#   Handle : cylinder r=0.015 m,  z = [0.05, 0.75]
#   Head   : box +/-0.12 x +/-0.03 x [0, 0.05] m
# Grasp point for IK: broom_world_pos + [0, 0, 0.55]  (upper handle)
BASE_XML = """
<mujoco model="sweep_scene">
  <compiler angle="radian"/>
  <option gravity="0 0 -9.81" timestep="0.002" iterations="50" noslip_iterations="10" cone="elliptic"/>
  
  <default>
    <!-- Increase global default friction (sliding, torsional, rolling) -->
    <geom friction="1.0 0.03 0.0001"/>
  </default>

  <asset>
    <mesh name="mug_mesh"        file="mug.obj"/>
    <mesh name="can_opener_mesh" file="can_opener.obj"/>
    <mesh name="action_fig_mesh" file="action_fig.obj"/>
    <mesh name="shoe_mesh"       file="shoe.obj"/>
    <mesh name="broom_mesh"      file="broom.obj"/>
  </asset>
  <worldbody>
    <light pos="0 0 2" dir="0 0 -1" directional="true" diffuse="0.8 0.8 0.8"/>
    <geom type="plane" size="1 1 0.1" rgba="0.9 0.9 0.9 1"/>
    <!-- Fixed overhead camera used by PerceptionModule -->
    <camera name="overhead_cam" pos="0 0 1.5" euler="0 0 0" fovy="45"/>
    <!-- Broom: freejoint so it can be grasped and lifted -->
    <body name="broom" pos="0.3 0.0 0.0">
      <freejoint name="broom_joint"/>
      <!-- Visual geom uses the mesh -->
      <geom name="broom_geom_viz" type="mesh" mesh="broom_mesh"
            rgba="0.65 0.38 0.12 1" contype="0" conaffinity="0"/>
      <!-- Collision geoms using primitives to avoid convex hull artifacts -->
      <!-- Handle: radius 0.015, height extent [0.05, 0.75] -->
      <!-- Center z = (0.75+0.05)/2 = 0.4. Half-length = (0.75-0.05)/2 = 0.35 -->
      <geom name="broom_col_handle" type="cylinder" size="0.015 0.35" pos="0 0 0.4"
            rgba="0 0 0 0" contype="1" conaffinity="3" condim="4" margin="0.003" gap="0.001"
            solref="0.001 1" solimp="0.9 0.95 0.001 0.5 2" friction="6.0 0.1 0.001"/>
      <!-- Head: box from x=-0.12..0.12, y=-0.03..0.03, z=0..0.05 -->
      <!-- Center z=0.025. Half-size: 0.12 0.03 0.025 -->
      <geom name="broom_col_head_body" type="box" size="0.12 0.03 0.024" pos="0 0 0.026"
            rgba="0 0 0 0" contype="1" conaffinity="3" condim="4" margin="0.003" gap="0.001"
            solref="0.001 1" solimp="0.9 0.95 0.001 0.5 2" friction="2.0 0.05 0.001"/>
      <geom name="broom_col_head_bottom" type="box" size="0.12 0.03 0.001" pos="0 0 0.001"
            rgba="0 0 0 0" contype="1" conaffinity="3" condim="4" margin="0.003" gap="0.001"
            solref="0.001 1" solimp="0.9 0.95 0.001 0.5 2" friction="12.0 0.3 0.003"/>
    </body>
    <!-- Sweepable objects -->
    <body name="mug"        pos="0    0    0.1"><freejoint/><geom type="mesh" mesh="mug_mesh"        rgba="0.8 0.2 0.2 1"/></body>
    <body name="can_opener" pos="0.1  0    0.1"><freejoint/><geom type="mesh" mesh="can_opener_mesh" rgba="0.2 0.8 0.2 1"/></body>
    <body name="action_fig" pos="-0.1 0    0.1"><freejoint/><geom type="mesh" mesh="action_fig_mesh" rgba="0.2 0.2 0.8 1"/></body>
    <body name="shoe"       pos="0    0.1  0.1"><freejoint/><geom type="mesh" mesh="shoe_mesh"       rgba="0.5 0.5 0.5 1"/></body>
  </worldbody>
</mujoco>
"""

# --- 3. Compose with MjSpec (Panda only — broom is already in BASE_XML) ---
scene_spec = mujoco.MjSpec.from_string(BASE_XML)
panda_spec = mujoco.MjSpec.from_file(PANDA_XML)

# We want torque control across the arm for RL, so let's modify the arm's actuators 
# from position/PD controllers ("general" with biasprm) to pure "motor" elements.
# The gripper (actuator 7) remains in position control so we just adjust the first 7.
for i in range(7):
    act = panda_spec.actuators[i]
    act.set_to_motor()
    act.biastype = mujoco.mjtBias.mjBIAS_NONE
    act.gaintype = mujoco.mjtGain.mjGAIN_FIXED
    act.gainprm = [1.0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
    act.biasprm = [0.0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
    
    # Apply standard Panda torque limits
    act.forcerange = [-87.0, 87.0] if i < 4 else [-12.0, 12.0]

# Increase the physical joint limits of the gripper for simulation (default max is 0.04 per finger)
for j in panda_spec.joints:
    if "finger_joint" in j.name:
        j.range = [0.0, 0.08] # double the opening distance to 8cm per finger (16cm total)

# The generic actuator8 handles the fingers via a split tendon. It is bound to 0-255 ctrlrange.
# We also need to double its implicit position target scaling.
# Original gain was 0.015686... meaning `target_pos = ctrl * 0.04 / 255`
panda_spec.actuators[-1].gainprm[0] = 0.08 / 255.0

# Add EE site to panda hand (menagerie panda.xml has no sites)
panda_hand = next(b for b in panda_spec.bodies if b.name == "hand")

# Disable the palm's single convex hull to allow deep grasping between fingers
for geom in panda_hand.geoms:
    if getattr(getattr(geom, "mesh", None), "name", "") == "hand_c":
        geom.contype = 0
        geom.conaffinity = 0

# Add large gripper plates to the fingers to increase grasp surface area
def find_body_in_spec(spec, name):
    for b in spec.bodies:
        if b.name == name: return b
    return None

for finger_name in ["left_finger", "right_finger"]:
    finger_body = find_body_in_spec(panda_spec, finger_name)
    if finger_body:
        # Check if we didn't already add one by accident
        plate = finger_body.add_geom()
        plate.type = mujoco.mjtGeom.mjGEOM_BOX
        # Local +Y is AWAY from the gap! The gap is in the -Y direction.
        # X is the vertical height of the fingers, Z is the forward reach.
        plate.size = [0.02, 0.004, 0.025]
        plate.pos = [0.0, 0.0, 0.045] # Shifted into the -Y gap
        plate.rgba = [0.2, 0.8, 0.2, 1] # Green plate to easily see it
        plate.friction = [100.0, 0.1, 0.001]
        plate.contype = 2
        plate.conaffinity = 1

        # Add "front" ridge (to stop broom sliding out the front)
        ridge1 = finger_body.add_geom()
        ridge1.type = mujoco.mjtGeom.mjGEOM_BOX
        ridge1.size = [0.02, 0.005, 0.004]  # Tall in X, thick in Y, narrow in Z
        ridge1.pos = [0.0, -0.009, 0.045 + 0.025]   # Front end of the plate (Z) and deep in gap (+Y)
        ridge1.rgba = [0.8, 0.2, 0.2, 1]     # Red to distinguish them
        ridge1.friction = [3.0, 0.1, 0.001]
        ridge1.contype = 2
        ridge1.conaffinity = 1

        # Add "back" ridge (to stop broom sliding back into the hand)
        ridge2 = finger_body.add_geom()
        ridge2.type = mujoco.mjtGeom.mjGEOM_BOX
        ridge2.size = [0.02, 0.005, 0.004]
        ridge2.pos = [0.0, -0.009, 0.045 - 0.025]   # Back end of the plate (Z)
        ridge2.rgba = [0.8, 0.2, 0.2, 1]
        ridge2.friction = [3.0, 0.1, 0.001]
        ridge2.contype = 2
        ridge2.conaffinity = 1

        # Stiffen contacts so the ridges resist interpenetration when the gripper closes.
        for geom in (plate, ridge1, ridge2):
            geom.margin = 0.003
            geom.gap = 0.001
            geom.condim = 4
            geom.solref = [0.005, 1.0]
            geom.solimp = [0.9, 0.95, 0.001, 0.5, 2.0]

ee_site      = panda_hand.add_site()
ee_site.name = "attachment_site"
ee_site.pos  = [0.0, 0.0, 0.1]   # ~10 cm toward fingertips in hand local frame

panda_frame       = scene_spec.worldbody.add_frame()
panda_frame.pos   = [0.7, 0.0, 0.0]
panda_frame.quat  = [0.0, 0.0, 0.0, 1.0] # face the table (pi around z)
scene_spec.attach(panda_spec, prefix="panda_", frame=panda_frame)

# --- 4. Compile ---------------------------------------------------------------
robot_model = scene_spec.compile()
robot_data  = mujoco.MjData(robot_model)
mujoco.mj_forward(robot_model, robot_data)

# --- 5. Resolve names ---------------------------------------------------------
site_names = [mujoco.mj_id2name(robot_model, mujoco.mjtObj.mjOBJ_SITE, i)
              for i in range(robot_model.nsite)]
print("Sites in scene:", site_names)

PANDA_JOINTS  = [f"panda_joint{i}" for i in range(1, 8)]
FINGER_JOINTS = ["panda_finger_joint1", "panda_finger_joint2"]
# Confirm EE_SITE matches one of the printed site names above
EE_SITE = "panda_attachment_site"

print(f"robot_model compiled -- nq: {robot_model.nq}, nv: {robot_model.nv}")


broom.obj already present
Sites in scene: ['panda_attachment_site']
robot_model compiled -- nq: 44, nv: 39


# 6. Retrain YOLO with Broom Class
Add the broom (class 1) to synthetic training data and retrain YOLOv8-OBB.

In [119]:
import os, cv2, yaml

if skip_perception_training:
    print("Skipping dataset generation and training for broom (using runs/obb/current).")
    BROOM_WEIGHTS = "runs/obb/current/weights/best.pt"
    print(f"BROOM_WEIGHTS: {BROOM_WEIGHTS}")
else:
    DATASET_DIR_R = "datasets/mujoco_obb_with_broom"
    os.makedirs(f"{DATASET_DIR_R}/images/train", exist_ok=True)
    os.makedirs(f"{DATASET_DIR_R}/labels/train", exist_ok=True)

    # Sweepable objects -> class 0, broom -> class 1
    classes_r = {
        "mug": 0, "can_opener": 0, "action_fig": 0, "shoe": 0,
        "broom": 1,
    }

    seg_renderer_r = mujoco.Renderer(robot_model, 480, 640)
    seg_renderer_r.enable_segmentation_rendering()
    renderer_r = mujoco.Renderer(robot_model, 480, 640)

    object_names_r = list(classes_r.keys())

    print("Generating synthetic data with broom. This will take a moment...")
    frames_to_generate = 100
    vis_img_r = None

    for i in range(frames_to_generate):
        for obj_name in object_names_r:
            body_id = mujoco.mj_name2id(robot_model, mujoco.mjtObj.mjOBJ_BODY, obj_name)
            rand_x, rand_y = np.random.uniform(-0.35, 0.35, size=2)

            # 80% chance to be relatively upright (z-axis rotation) and 20% fully random
            if np.random.rand() > 0.2:
                rand_yaw = np.random.uniform(0, 2 * np.pi)
                q_rand = np.array([np.cos(rand_yaw/2), 0.0, 0.0, np.sin(rand_yaw/2)])
            else:
                q_rand = np.random.randn(4)
                q_rand /= np.linalg.norm(q_rand)

            qpos_start = robot_model.jnt_qposadr[robot_model.body_jntadr[body_id]]
            robot_data.qpos[qpos_start:qpos_start + 3] = [rand_x, rand_y, 0.15]
            robot_data.qpos[qpos_start + 3:qpos_start + 7] = q_rand

        mujoco.mj_forward(robot_model, robot_data)
        for _ in range(np.random.randint(10, 60)):
            mujoco.mj_step(robot_model, robot_data)

        renderer_r.update_scene(robot_data, camera="overhead_cam")
        rgb_img = renderer_r.render()

        seg_renderer_r.update_scene(robot_data, camera="overhead_cam")
        seg_img = seg_renderer_r.render()

        img_h, img_w = rgb_img.shape[:2]
        if i == frames_to_generate - 1:
            vis_img_r = rgb_img.copy()

        label_file = open(f"{DATASET_DIR_R}/labels/train/frame_{i}.txt", "w")

        for obj_name, class_idx in classes_r.items():
            body_id = mujoco.mj_name2id(robot_model, mujoco.mjtObj.mjOBJ_BODY, obj_name)
            geom_start = robot_model.body_geomadr[body_id]
            geom_num = robot_model.body_geomnum[body_id]
            mask = np.isin(seg_img[:, :, 0], range(geom_start, geom_start + geom_num)).astype(np.uint8) * 255
            contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            if contours:
                c = max(contours, key=cv2.contourArea)
                if cv2.contourArea(c) > 50:
                    rect = cv2.minAreaRect(c)
                    box = cv2.boxPoints(rect)
                    if vis_img_r is not None:
                        cv2.drawContours(vis_img_r, [np.int32(box)], 0, (0, 255, 0), 2)
                    box[:, 0] /= img_w
                    box[:, 1] /= img_h
                    label_str = " ".join([f"{coord:.5f}" for coordinate in box for coord in coordinate])
                    label_file.write(f"{class_idx} {label_str}\n")

        label_file.close()
        cv2.imwrite(f"{DATASET_DIR_R}/images/train/frame_{i}.jpg",
                    cv2.cvtColor(rgb_img, cv2.COLOR_RGB2BGR))

    print("Dataset generation complete!")
    if vis_img_r is not None:
        media.show_image(vis_img_r)

    # Write YAML and retrain
    yaml_config_r = {
        "path": os.path.abspath(DATASET_DIR_R),
        "train": "images/train",
        "val":   "images/train",
        "names": {0: "object", 1: "broom"},
    }
    with open("mujoco_dataset_with_broom.yaml", "w") as f:
        yaml.dump(yaml_config_r, f)

    model_obb_r = YOLO("yolov8n-obb.pt")
    results_r = model_obb_r.train(
        data="mujoco_dataset_with_broom.yaml",
        epochs=50, imgsz=640, device="cpu", verbose=False,
        lr0=0.001, batch=16
    )
    print("Training finished!")

    # Automatically resolve the correct weights path from the completed training run
    BROOM_WEIGHTS = str(results_r.save_dir / "weights" / "best.pt")
    print(f"BROOM_WEIGHTS: {BROOM_WEIGHTS}")

Skipping dataset generation and training for broom (using runs/obb/current).
BROOM_WEIGHTS: runs/obb/current/weights/best.pt


# 6. Grasp Planning

In [120]:
import numpy as np
import mujoco
import cv2
import time

PANDA_READY = np.array([0, -np.pi / 4, 0, -3 * np.pi / 4, 0, np.pi / 2, np.pi / 4], dtype=np.float32)

_IK_WINDOW_NAME = "IK Preview"
_ik_window_ready = False

# --- Joint limits for IK projection ---
def _get_joint_limits(model, joint_names):
    limits = []
    for name in joint_names:
        j_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, name)
        lo, hi = model.jnt_range[j_id]
        if lo == 0.0 and hi == 0.0:
            lo, hi = -3.0, 3.0
        limits.append((float(lo), float(hi)))
    return np.array(limits, dtype=np.float32)

JOINT_LIMITS = _get_joint_limits(robot_model, PANDA_JOINTS)

def _clip_q(q, limits):
    return np.minimum(np.maximum(q, limits[:, 0]), limits[:, 1])

def _show_marker(renderer, data, marker_pos, camera="overhead_cam", radius=0.05, rgba=(1, 0, 1, 0.9)):
    renderer.update_scene(data, camera=camera)
    scene = renderer.scene
    if scene.ngeom < scene.maxgeom:
        mujoco.mjv_initGeom(
            scene.geoms[scene.ngeom],
            mujoco.mjtGeom.mjGEOM_SPHERE, np.array([radius, 0, 0]),
            marker_pos, np.eye(3).flatten(), np.array(rgba, dtype=np.float32)
        )
        scene.geoms[scene.ngeom].category = mujoco.mjtCatBit.mjCAT_DECOR
        scene.ngeom += 1
    rgb = renderer.render()
    bgr = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)
    cv2.imshow(_IK_WINDOW_NAME, bgr)
    cv2.waitKey(1)

# --- Reset scene to model defaults ---
def reset_scene(model, data, arm_qpos=None):
    mujoco.mj_resetData(model, data)
    data.qpos[:] = model.qpos0
    data.qvel[:] = 0.0
    if model.nu > 0:
        data.ctrl[:] = 0.0
    if arm_qpos is not None:
        jnt_ids = [mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, n) for n in PANDA_JOINTS]
        qpos_adr = [model.jnt_qposadr[j] for j in jnt_ids]
        for i, adr in enumerate(qpos_adr):
            data.qpos[adr] = arm_qpos[i]
    mujoco.mj_forward(model, data)

# --- Simple settle step to let objects fall ---
def settle_scene(model, data, arm_qpos=None, steps=200):
    jnt_ids = [mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, n) for n in PANDA_JOINTS]
    qpos_adr = [model.jnt_qposadr[j] for j in jnt_ids]
    dof_adr = [model.jnt_dofadr[j] for j in jnt_ids]
    for _ in range(steps):
        if arm_qpos is not None:
            for i, adr in enumerate(qpos_adr):
                data.qpos[adr] = arm_qpos[i]
            for d in dof_adr:
                data.qvel[d] = 0.0
        mujoco.mj_step(model, data)
    mujoco.mj_forward(model, data)

# --- Position + orientation IK (damped least squares) ---
def solve_ik_to_site(
    model,
    data,
    site_name,
    target_pos,
    q_init,
    target_rot=None,
    max_iters=400,
    tol=2e-3,
    rot_tol=5e-2,
    damp=5e-3,
    rot_w=0.2,
 ):
    jnt_ids = [mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, n) for n in PANDA_JOINTS]
    qpos_adr = [model.jnt_qposadr[j] for j in jnt_ids]
    dof_adr = [model.jnt_dofadr[j] for j in jnt_ids]
    site_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_SITE, site_name)

    target_pos = target_pos.copy()
    target_pos[2] = max(target_pos[2], 0.05)

    q = q_init.copy()
    for _ in range(max_iters):
        q = _clip_q(q, JOINT_LIMITS)
        for i, adr in enumerate(qpos_adr):
            data.qpos[adr] = q[i]
        mujoco.mj_forward(model, data)

        cur = data.site_xpos[site_id].copy()
        err_pos = target_pos - cur
        if np.linalg.norm(err_pos) > 0.1:
            err_pos = err_pos / np.linalg.norm(err_pos) * 0.1

        jacp = np.zeros((3, model.nv))
        jacr = np.zeros((3, model.nv))
        mujoco.mj_jacSite(model, data, jacp, jacr, site_id)

        rot_err = None
        if target_rot is not None:
            cur_mat = data.site_xmat[site_id].reshape(3, 3)
            quat_cur = np.zeros(4, dtype=np.float64)
            quat_tgt = np.zeros(4, dtype=np.float64)
            mujoco.mju_mat2Quat(quat_cur, cur_mat.flatten())
            mujoco.mju_mat2Quat(quat_tgt, target_rot.flatten())
            quat_err = np.zeros(4, dtype=np.float64)
            quat_cur_conj = quat_cur.copy()
            quat_cur_conj[1:] *= -1.0
            mujoco.mju_mulQuat(quat_err, quat_tgt, quat_cur_conj)
            rot_err = np.zeros(3, dtype=np.float64)
            mujoco.mju_quat2Vel(rot_err, quat_err, 1.0)
            if np.linalg.norm(rot_err) > 0.5:
                rot_err = rot_err / np.linalg.norm(rot_err) * 0.5

        if np.linalg.norm(err_pos) < tol and (rot_err is None or np.linalg.norm(rot_err) < rot_tol):
            return q, True

        if rot_err is not None:
            err = np.concatenate([err_pos, rot_w * rot_err])
            J = np.vstack([jacp[:, dof_adr], rot_w * jacr[:, dof_adr]])
            A = J @ J.T + damp * np.eye(6)
        else:
            err = err_pos
            J = jacp[:, dof_adr]
            A = J @ J.T + damp * np.eye(3)

        dq = J.T @ np.linalg.solve(A, err)
        q = q + 0.5 * dq

    return q, False

def _get_arm_actuator_ids(model, joint_names):
    jnt_ids = [mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, n) for n in joint_names]
    jnt_to_idx = {jid: i for i, jid in enumerate(jnt_ids)}
    act_ids = []
    for act_id in range(model.nu):
        joint_id = int(model.actuator_trnid[act_id, 0])
        if joint_id in jnt_to_idx:
            act_ids.append((jnt_to_idx[joint_id], act_id))
    act_ids.sort(key=lambda x: x[0])
    return [act_id for _, act_id in act_ids], jnt_ids

def _find_gripper_actuators_by_name(model):
    act_ids = []
    for act_id in range(model.nu):
        act_name = mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_ACTUATOR, act_id)
        if act_name and ("finger" in act_name or "gripper" in act_name):
            act_ids.append(act_id)
    return act_ids

def _find_gripper_actuators_by_tendon(model):
    act_ids = []
    for act_id in range(model.nu):
        if int(model.actuator_trntype[act_id]) != int(mujoco.mjtTrn.mjTRN_TENDON):
            continue
        tendon_id = int(model.actuator_trnid[act_id, 0])
        tendon_name = mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_TENDON, tendon_id)
        if tendon_name and ("finger" in tendon_name or "gripper" in tendon_name or "split" in tendon_name):
            act_ids.append(act_id)
    return act_ids

def _get_gripper_actuator_ids(model, joint_names):
    hardcoded = ["panda_actuator8"]
    hardcoded_ids = [
        mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_ACTUATOR, name)
        for name in hardcoded
    ]
    if all(aid >= 0 for aid in hardcoded_ids):
        return hardcoded_ids

    tendon_act_ids = _find_gripper_actuators_by_tendon(model)
    if tendon_act_ids:
        return tendon_act_ids

    jnt_ids = [mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, n) for n in joint_names]
    jnt_to_idx = {jid: i for i, jid in enumerate(jnt_ids)}
    act_ids = []
    for act_id in range(model.nu):
        joint_id = int(model.actuator_trnid[act_id, 0])
        if joint_id in jnt_to_idx:
            act_ids.append((jnt_to_idx[joint_id], act_id))
    act_ids.sort(key=lambda x: x[0])
    joint_act_ids = [act_id for _, act_id in act_ids]
    if joint_act_ids:
        return joint_act_ids

    name_act_ids = _find_gripper_actuators_by_name(model)
    if name_act_ids:
        return name_act_ids

    # Fall back to last two actuators (Panda gripper in menagerie).
    if model.nu >= 2:
        return [model.nu - 2, model.nu - 1]
    return []

def _set_gripper_qpos(model, data, joint_names, value):
    for name in joint_names:
        j_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, name)
        if j_id < 0:
            continue
        adr = model.jnt_qposadr[j_id]
        data.qpos[adr] = float(value)
    mujoco.mj_forward(model, data)

def _debug_actuators(model, act_ids):
    info = []
    for act_id in act_ids:
        act_name = mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_ACTUATOR, act_id)
        joint_id = int(model.actuator_trnid[act_id, 0])
        joint_name = mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_JOINT, joint_id)
        gear = float(model.actuator_gear[act_id, 0])
        lo, hi = model.actuator_forcerange[act_id]
        info.append((act_id, act_name, joint_name, gear, (float(lo), float(hi))))
    print("Actuators driving arm:", info)
    print("Total actuators:", model.nu)

def _set_torque_limits(model, act_ids, arm_limits, wrist_limits):
    for i, act_id in enumerate(act_ids):
        limit = arm_limits if i < 4 else wrist_limits
        model.actuator_forcerange[act_id, 0] = -float(limit)
        model.actuator_forcerange[act_id, 1] = float(limit)
    print("Updated actuator forcerange:", [tuple(model.actuator_forcerange[a]) for a in act_ids])

def _apply_direct_torque(data, dof_adr, tau):
    data.qfrc_applied[:] = 0.0
    for i, d in enumerate(dof_adr):
        data.qfrc_applied[d] = float(tau[i])

# --- Execute joint path with PD torque control ---
def execute_joint_path(
    model,
    data,
    path,
    renderer,
    camera="overhead_cam",
    hold_steps=8,
    sleep_s=0.0,
    kp=800.0,
    kd=30.0,
    wait_on_exit=True,
    exit_key="q",
    target_pos=None,
    marker_pos=None,
    goal_pos=None,
    use_direct_torque=False,
    render_stride=3,
    gripper_ctrl=None,
    reset_start_pose=True,
    ramp_steps=25,
    ramp_start=0.2,
    overlay_fn=None,
    overlay_origin=(10, 24),
    overlay_line_height=22,
    overlay_color=(255, 255, 255),
 ):
    if path is None or len(path) == 0:
        print("No path to execute.")
        return

    act_ids, jnt_ids = _get_arm_actuator_ids(model, PANDA_JOINTS)
    if not act_ids:
        print("No arm actuators found; cannot run torque control.")
        return
    _debug_actuators(model, act_ids)
    _set_torque_limits(model, act_ids, arm_limits=200.0, wrist_limits=80.0)
    gripper_act_ids = _get_gripper_actuator_ids(model, FINGER_JOINTS)

    qpos_adr = [model.jnt_qposadr[j] for j in jnt_ids]
    dof_adr = [model.jnt_dofadr[j] for j in jnt_ids]
    print("Arm dof adr:", dof_adr)

    path = [np.array(p, dtype=np.float32) for p in path]
    if not reset_start_pose:
        q_cur = np.array([data.qpos[a] for a in qpos_adr], dtype=np.float32)
        if len(path) == 0 or np.max(np.abs(path[0] - q_cur)) > 1e-4:
            path = [q_cur] + path

    def _apply_gripper_ctrl(step_idx):
        if gripper_ctrl is None or not gripper_act_ids:
            return
        if isinstance(gripper_ctrl, (tuple, list, np.ndarray)) and len(gripper_ctrl) == 3:
            g0, g1, steps = gripper_ctrl
            steps = int(steps)
            if steps <= 1:
                g = float(g1)
            else:
                t = min(1.0, float(step_idx) / float(steps - 1))
                g = float(g0) + (float(g1) - float(g0)) * t
        else:
            g = float(gripper_ctrl)
        for act_id in gripper_act_ids:
            data.ctrl[act_id] = g

    def _ramp_scale(step_idx):
        if ramp_steps <= 0:
            return 1.0
        t = min(1.0, float(step_idx) / float(ramp_steps))
        return ramp_start + (1.0 - ramp_start) * t

    def _draw_overlay(img, lines):
        if not lines:
            return
        x, y = overlay_origin
        for line in lines:
            text = str(line)
            (tw, th), baseline = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
            rect_pt1 = (int(x - 6), int(y - th - 6))
            rect_pt2 = (int(x + tw + 6), int(y + baseline + 6))
            cv2.rectangle(img, rect_pt1, rect_pt2, (0, 0, 0), -1)
            cv2.putText(
                img,
                text,
                (int(x), int(y)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                overlay_color,
                2,
                cv2.LINE_AA,
            )
            y += overlay_line_height

    if reset_start_pose:
        for i, adr in enumerate(qpos_adr):
            data.qpos[adr] = path[0][i]
        for d in dof_adr:
            data.qvel[d] = 0.0
        data.ctrl[:] = 0.0
        _apply_gripper_ctrl(0)
        mujoco.mj_forward(model, data)
    else:
        _apply_gripper_ctrl(0)
        mujoco.mj_forward(model, data)

    # Hold the start pose briefly to confirm torque control is active
    step_idx = 0
    for _ in range(30):
        scale = _ramp_scale(step_idx)
        step_idx += 1
        q = np.array([data.qpos[a] for a in qpos_adr], dtype=np.float32)
        qd = np.array([data.qvel[a] for a in dof_adr], dtype=np.float32)
        tau = (kp * scale) * (path[0] - q) - (kd * scale) * qd + data.qfrc_bias[dof_adr]
        if use_direct_torque:
            _apply_direct_torque(data, dof_adr, tau)
        else:
            for i, act_id in enumerate(act_ids):
                lo, hi = model.actuator_forcerange[act_id]
                data.ctrl[act_id] = float(np.clip(tau[i], lo, hi))
        _apply_gripper_ctrl(step_idx)
        mujoco.mj_step(model, data)
    print("Hold max |tau|:", float(np.max(np.abs(tau))))
    mujoco.mj_forward(model, data)

    global _ik_window_ready
    if (not _ik_window_ready or
        cv2.getWindowProperty(_IK_WINDOW_NAME, cv2.WND_PROP_VISIBLE) < 1):
        cv2.namedWindow(_IK_WINDOW_NAME, cv2.WINDOW_NORMAL)
        _ik_window_ready = True

    render_count = 0
    try:
        for q_target in path:
            for _ in range(hold_steps):
                scale = _ramp_scale(step_idx)
                step_idx += 1
                q = np.array([data.qpos[a] for a in qpos_adr], dtype=np.float32)
                qd = np.array([data.qvel[a] for a in dof_adr], dtype=np.float32)
                tau = (kp * scale) * (q_target - q) - (kd * scale) * qd + data.qfrc_bias[dof_adr]
                if use_direct_torque:
                    _apply_direct_torque(data, dof_adr, tau)
                else:
                    for i, act_id in enumerate(act_ids):
                        lo, hi = model.actuator_forcerange[act_id]
                        data.ctrl[act_id] = float(np.clip(tau[i], lo, hi))
                _apply_gripper_ctrl(step_idx)
                mujoco.mj_step(model, data)
                render_count += 1
                if render_count % render_stride != 0:
                    continue
                renderer.update_scene(data, camera=camera)

                scene = renderer.scene
                if target_pos is not None and scene.ngeom < scene.maxgeom:
                    mujoco.mjv_initGeom(
                        scene.geoms[scene.ngeom],
                        mujoco.mjtGeom.mjGEOM_SPHERE, np.array([0.05, 0, 0]),
                        target_pos, np.eye(3).flatten(), np.array([1, 0, 1, 0.7], dtype=np.float32)
                    )
                    scene.geoms[scene.ngeom].category = mujoco.mjtCatBit.mjCAT_DECOR
                    scene.ngeom += 1

                if marker_pos is not None and scene.ngeom < scene.maxgeom:
                    mujoco.mjv_initGeom(
                        scene.geoms[scene.ngeom],
                        mujoco.mjtGeom.mjGEOM_SPHERE, np.array([0.05, 0, 0]),
                        marker_pos, np.eye(3).flatten(), np.array([1, 0, 1, 0.7], dtype=np.float32)
                    )
                    scene.geoms[scene.ngeom].category = mujoco.mjtCatBit.mjCAT_DECOR
                    scene.ngeom += 1

                if goal_pos is not None and scene.ngeom < scene.maxgeom:
                    mujoco.mjv_initGeom(
                        scene.geoms[scene.ngeom],
                        mujoco.mjtGeom.mjGEOM_SPHERE, np.array([0.2, 0, 0]),
                        goal_pos, np.eye(3).flatten(), np.array([0, 1, 0, 0.35], dtype=np.float32)
                    )
                    scene.geoms[scene.ngeom].category = mujoco.mjtCatBit.mjCAT_DECOR
                    scene.ngeom += 1

                rgb = renderer.render()
                bgr = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)
                if overlay_fn is not None:
                    lines = overlay_fn(model, data)
                    _draw_overlay(bgr, lines)
                cv2.imshow(_IK_WINDOW_NAME, bgr)
                key = cv2.waitKey(1) & 0xFF
                if key == ord(exit_key):
                    cv2.destroyWindow(_IK_WINDOW_NAME)
                    _ik_window_ready = False
                    return
                if key == 27:
                    return
                if sleep_s > 0.0:
                    time.sleep(sleep_s)
    finally:
        if wait_on_exit:
            while True:
                key = cv2.waitKey(30) & 0xFF
                if key == ord(exit_key) or key == 27:
                    break
            cv2.destroyWindow(_IK_WINDOW_NAME)
            _ik_window_ready = False
        else:
            for _ in range(5):
                cv2.waitKey(1)

In [121]:
def _pick_broom_detection(detections):
    # it should prefer the broom class when present; otherwise return the most confident object.
    if not detections:
        return None
    broom_dets = [d for d in detections if d.get("class") == "broom"]
    candidates = broom_dets if broom_dets else detections
    return max(candidates, key=lambda d: float(d.get("confidence", 0.0)))

def _pick_sweepable_detection(detections):
    # Pick a non-broom object with the highest confidence.
    if not detections:
        return None
    candidates = [d for d in detections if d.get("class") != "broom"]
    if not candidates:
        return None
    return max(candidates, key=lambda d: float(d.get("confidence", 0.0)))

In [122]:
def _camera_to_world(model, cam_name, pos_cam):
    cam_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_CAMERA, cam_name)
    cam_pos = model.cam_pos[cam_id].copy()
    cam_quat = model.cam_quat[cam_id].copy()
    cam_mat = np.zeros(9, dtype=np.float64)
    mujoco.mju_quat2Mat(cam_mat, cam_quat)
    cam_R = cam_mat.reshape(3, 3)
    return cam_pos + cam_R @ pos_cam

In [123]:
# Demo setup: use perceived broom handle as target
if "perceptor_broom" not in globals():
    perceptor_broom = PerceptionModule(BROOM_WEIGHTS)

sweepable_names = ["mug", "can_opener", "action_fig", "shoe"]

panda_base_xy = np.array([0.2, 0.0], dtype=np.float32)
broom_target_xy = np.array([0.15, -0.25], dtype=np.float32)

goal_xy = np.array([0.1, 0.0], dtype=np.float32)
goal_radius = 0.2
goal_pos = np.array([goal_xy[0], goal_xy[1], 0.02], dtype=np.float32)

divider_normal = broom_target_xy - panda_base_xy
divider_normal = divider_normal / (np.linalg.norm(divider_normal) + 1e-8)
divider_margin = 0.05


def _enforce_divider(xy):
    dot = float(np.dot(xy - panda_base_xy, divider_normal))
    if dot > -divider_margin:
        xy = xy - (dot + divider_margin) * divider_normal
    return xy


def _yaw_to_quat(yaw):
    return np.array([np.cos(yaw / 2.0), 0.0, 0.0, np.sin(yaw / 2.0)], dtype=np.float32)


def _set_body_pose(model, data, body_name, xy, z=0.15, yaw=0.0):
    body_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, body_name)
    if body_id < 0:
        return False
    qpos_start = model.jnt_qposadr[model.body_jntadr[body_id]]
    data.qpos[qpos_start + 0] = float(xy[0])
    data.qpos[qpos_start + 1] = float(xy[1])
    data.qpos[qpos_start + 2] = float(z)
    quat = _yaw_to_quat(yaw)
    data.qpos[qpos_start + 3:qpos_start + 7] = quat
    return True


def _place_objects_line(model, data, obj_names, start_xy, direction_xy, spacing):
    direction_xy = direction_xy / (np.linalg.norm(direction_xy) + 1e-8)
    for i, name in enumerate(obj_names):
        pos_xy = start_xy + direction_xy * (spacing * i)
        pos_xy = _enforce_divider(pos_xy)
        yaw = float(i) * 0.3
        _set_body_pose(model, data, name, pos_xy, yaw=yaw)


def _place_objects_spread(model, data, obj_names):
    positions = [
        np.array([0.32, 0.22], dtype=np.float32),
        np.array([0.28, -0.28], dtype=np.float32),
        np.array([-0.22, 0.20], dtype=np.float32),
        np.array([-0.12, -0.30], dtype=np.float32),
    ]
    for name, pos in zip(obj_names, positions):
        pos_xy = _enforce_divider(pos)
        yaw = float(np.random.uniform(-np.pi, np.pi))
        _set_body_pose(model, data, name, pos_xy, yaw=yaw)


def _place_objects_cluster(model, data, obj_names):
    center = np.array([0.32, -0.25], dtype=np.float32)
    for name in obj_names:
        jitter = np.random.uniform(-0.02, 0.02, size=2)
        pos_xy = _enforce_divider(center + jitter)
        yaw = float(np.random.uniform(-np.pi, np.pi))
        _set_body_pose(model, data, name, pos_xy, yaw=yaw)


def _place_objects_line_far(model, data, obj_names):
    start_xy = panda_base_xy + np.array([0.14, -0.12], dtype=np.float32)
    direction_xy = np.array([0.0, 1.0], dtype=np.float32)
    _place_objects_line(model, data, obj_names, start_xy, direction_xy, spacing=0.07)


def _place_objects_circle(model, data, obj_names):
    radius = 0.2
    for i, name in enumerate(obj_names):
        theta = (2.0 * np.pi / len(obj_names)) * i
        pos_xy = goal_xy + radius * np.array([np.cos(theta), np.sin(theta)], dtype=np.float32)
        pos_xy = _enforce_divider(pos_xy)
        yaw = float(theta)
        _set_body_pose(model, data, name, pos_xy, yaw=yaw)


def _get_object_xy(model, data, obj_name):
    body_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, obj_name)
    if body_id < 0:
        return None
    return data.xpos[body_id][:2].copy()


def _count_in_goal(model, data, obj_names, goal_xy, radius):
    count = 0
    for name in obj_names:
        pos_xy = _get_object_xy(model, data, name)
        if pos_xy is None:
            continue
        if np.linalg.norm(pos_xy - goal_xy) <= radius:
            count += 1
    return count


def _object_centroid(model, data, obj_names):
    points = []
    for name in obj_names:
        pos_xy = _get_object_xy(model, data, name)
        if pos_xy is not None:
            points.append(pos_xy)
    if not points:
        return np.array([np.nan, np.nan], dtype=np.float32)
    return np.mean(np.stack(points, axis=0), axis=0)


def _demo_stats_lines(model, data, obj_names, goal_xy, radius, broom_body_id, ee_site_id):
    count_in_goal = _count_in_goal(model, data, obj_names, goal_xy, radius)
    centroid = _object_centroid(model, data, obj_names)
    broom_pos = data.xpos[broom_body_id][:2]
    ee_pos = data.site_xpos[ee_site_id][:2]
    return [
        f"Objects in goal: {count_in_goal}/{len(obj_names)}",
        f"Centroid: [{centroid[0]:.3f}, {centroid[1]:.3f}]",
        f"Broom head: [{broom_pos[0]:.3f}, {broom_pos[1]:.3f}]",
        f"EE: [{ee_pos[0]:.3f}, {ee_pos[1]:.3f}]",
    ]


def run_demo_case(case_name, place_fn):
    arm_seed = PANDA_READY
    reset_scene(robot_model, robot_data, arm_qpos=arm_seed)

    broom_body_id = mujoco.mj_name2id(robot_model, mujoco.mjtObj.mjOBJ_BODY, "broom")
    broom_jnt_adr = robot_model.jnt_qposadr[robot_model.body_jntadr[broom_body_id]]
    broom_z = float(robot_data.qpos[broom_jnt_adr + 2])
    robot_data.qpos[broom_jnt_adr + 0] = float(broom_target_xy[0])
    robot_data.qpos[broom_jnt_adr + 1] = float(broom_target_xy[1])
    robot_data.qpos[broom_jnt_adr + 2] = broom_z

    place_fn(robot_model, robot_data, sweepable_names)
    settle_scene(robot_model, robot_data, arm_qpos=arm_seed, steps=200)

    gripper_act_ids = _get_gripper_actuator_ids(robot_model, FINGER_JOINTS)
    gripper_open = 0.04
    gripper_closed = 0.0
    gripper_available = bool(gripper_act_ids)
    if gripper_available:
        ctrl_lo, ctrl_hi = robot_model.actuator_ctrlrange[gripper_act_ids[0]]
        gripper_closed = float(ctrl_lo)
        gripper_open = float(ctrl_hi)
        gripper_preopen = gripper_open * 0.6

        for act_id in gripper_act_ids:
            robot_model.actuator_forcerange[act_id, 0] = -500.0
            robot_model.actuator_forcerange[act_id, 1] = 500.0
            robot_model.actuator_biasprm[act_id, 1] = -5000.0
            robot_model.actuator_biasprm[act_id, 2] = -50.0
            if ctrl_hi > 0:
                robot_model.actuator_gainprm[act_id, 0] = 0.06 * 5000.0 / ctrl_hi

        for _ in range(30):
            for act_id in gripper_act_ids:
                robot_data.ctrl[act_id] = float(gripper_preopen)
            mujoco.mj_step(robot_model, robot_data)
    else:
        print("Warning: no gripper actuators found; cannot open gripper.")

    renderer_t = mujoco.Renderer(robot_model, 480, 640)
    depth_renderer_t = mujoco.Renderer(robot_model, 480, 640)
    depth_renderer_t.enable_depth_rendering()

    renderer_t.update_scene(robot_data, camera="overhead_cam")
    depth_renderer_t.update_scene(robot_data, camera="overhead_cam")
    img_rgb = renderer_t.render()
    img_depth = depth_renderer_t.render()

    detected_objects, _ = perceptor_broom.get_transforms(img_rgb, img_depth)
    broom_det = _pick_broom_detection(detected_objects)
    sweep_obj_det = _pick_sweepable_detection(detected_objects)

    if broom_det is None:
        print("No broom detected; check weights or camera framing.")
        return False

    broom_handle_cam = broom_det["position_camera_frame"].astype(np.float32)
    broom_handle_world = _camera_to_world(robot_model, "overhead_cam", broom_handle_cam)

    mirror_y = False
    mirror_z_about_cam = False

    cam_id = mujoco.mj_name2id(robot_model, mujoco.mjtObj.mjOBJ_CAMERA, "overhead_cam")
    cam_pos = robot_model.cam_pos[cam_id].copy()
    if mirror_y:
        broom_handle_world[1] = -broom_handle_world[1]
    if mirror_z_about_cam:
        broom_handle_world[2] = 2.0 * cam_pos[2] - broom_handle_world[2]

    rot_cam = broom_det.get("rotation_camera_frame")
    if rot_cam is None:
        rot_cam = _rotation_from_obb(broom_det.get("rotation_rad", 0.0))
    rot_cam = rot_cam @ _rotation_z_offset(-np.pi / 4.0)
    rot_world = _camera_to_world_rotation(robot_model, "overhead_cam", rot_cam)
    if mirror_y:
        rot_world = np.diag([1.0, -1.0, 1.0]).astype(np.float32) @ rot_world
    if mirror_z_about_cam:
        rot_world = np.diag([1.0, 1.0, -1.0]).astype(np.float32) @ rot_world

    rot_world = rot_world @ _rotation_y_90()

    approach_offset = np.array([0.0, 0.08, 0.0], dtype=np.float32)
    target_pos = broom_handle_world.copy()
    target_rot = rot_world.copy()
    grasp_lateral_offset = 0.015
    target_pos = target_pos + target_rot[:, 1] * grasp_lateral_offset
    pregrasp_pos = target_pos + approach_offset
    site_id = mujoco.mj_name2id(robot_model, mujoco.mjtObj.mjOBJ_SITE, EE_SITE)

    obj_names_present = [
        name for name in sweepable_names
        if mujoco.mj_name2id(robot_model, mujoco.mjtObj.mjOBJ_BODY, name) >= 0
    ]

    status = {"done": False, "success": False}

    def overlay_fn(model, data):
        lines = _demo_stats_lines(
            model, data, obj_names_present, goal_xy, goal_radius, broom_body_id, site_id
        )
        if status["done"]:
            lines.append("Sweep success: YES" if status["success"] else "Sweep success: NO")
        return lines

    q_start = np.array([
        robot_data.qpos[robot_model.jnt_qposadr[mujoco.mj_name2id(
            robot_model, mujoco.mjtObj.mjOBJ_JOINT, n
        )]] for n in PANDA_JOINTS
    ])
    if gripper_available:
        for _ in range(20):
            for act_id in gripper_act_ids:
                robot_data.ctrl[act_id] = float(gripper_open)
            mujoco.mj_step(robot_model, robot_data)

    def _move_to_target(model, data, target_pos, target_rot, q_start, renderer, gripper_ctrl, steps=40, reset_start_pose=False, marker_pos=None, goal_pos=None, overlay_fn=None):
        q_target, ok = _solve_ik_preserve_state(
            model, data, EE_SITE, target_pos, q_start, target_rot=target_rot
        )
        if not ok:
            return q_start, False

        path = [(1.0 - t) * q_start + t * q_target for t in np.linspace(0.0, 1.0, steps)]
        execute_joint_path(
            model, data, path, renderer,
            target_pos=target_pos, marker_pos=marker_pos, goal_pos=goal_pos, use_direct_torque=True, render_stride=3,
            gripper_ctrl=gripper_ctrl, wait_on_exit=False,
            reset_start_pose=reset_start_pose,
            overlay_fn=overlay_fn,
        )
        return q_target, True

    q_pre, ok_pre = _solve_ik_preserve_state(
        robot_model, robot_data, EE_SITE, pregrasp_pos, q_start, target_rot=target_rot
    )
    if not ok_pre:
        print("Failed to reach pregrasp; try a different pose")
        return False

    steps = 30
    path_pre = [(1.0 - t) * q_start + t * q_pre for t in np.linspace(0.0, 1.0, steps)]
    ik_renderer = mujoco.Renderer(robot_model, 480, 640)
    execute_joint_path(
        robot_model, robot_data, path_pre, ik_renderer,
        target_pos=pregrasp_pos, goal_pos=goal_pos, use_direct_torque=True, render_stride=3,
        gripper_ctrl=gripper_open if gripper_available else None,
        wait_on_exit=False,
        overlay_fn=overlay_fn,
    )

    q_goal, ok = _move_to_target(
        robot_model, robot_data, target_pos, target_rot, q_pre, ik_renderer,
        gripper_open if gripper_available else None, steps=20, goal_pos=goal_pos, overlay_fn=overlay_fn
    )
    if not ok:
        print("Failed to reach perceived target; try a different pose")
        return False

    if gripper_available:
        print("Closing gripper...")
        execute_joint_path(
            robot_model, robot_data, [q_goal], ik_renderer,
            target_pos=target_pos, goal_pos=goal_pos, use_direct_torque=True, render_stride=3,
            gripper_ctrl=(gripper_open, gripper_closed, 80), hold_steps=80,
            wait_on_exit=False,
            overlay_fn=overlay_fn,
        )
        for _ in range(50):
            for act_id in gripper_act_ids:
                robot_data.ctrl[act_id] = float(gripper_closed)
            mujoco.mj_step(robot_model, robot_data)
        print("Gripper closed!")

        print("Attempting to lift broom to test friction...")
        lift_pos = target_pos + np.array([0.0, 0.0, 0.3], dtype=np.float32)
        q_lift, ok_lift = _move_to_target(
            robot_model, robot_data, lift_pos, target_rot, q_goal, ik_renderer,
            gripper_closed, steps=40, goal_pos=goal_pos, overlay_fn=overlay_fn
        )
        if not ok_lift:
            print("IK failed for lift.")
            return False

        execute_joint_path(
            robot_model, robot_data, [q_lift], ik_renderer,
            target_pos=lift_pos, goal_pos=goal_pos, use_direct_torque=True, render_stride=10,
            gripper_ctrl=gripper_closed, hold_steps=100, wait_on_exit=False,
            reset_start_pose=False,
            overlay_fn=overlay_fn,
        )

        print("Moving broom to sweeping height...")
        sweeping_z_offset = 0.24
        sweeping_pos = lift_pos.copy()
        sweeping_pos[2] = sweeping_z_offset + (lift_pos[2] - target_pos[2])

        q_sweep, ok_sweep = _move_to_target(
            robot_model, robot_data, sweeping_pos, target_rot, q_lift, ik_renderer,
            gripper_closed, steps=40, reset_start_pose=False, goal_pos=goal_pos, overlay_fn=overlay_fn
        )
        if not ok_sweep:
            print("IK failed for moving to sweeping height.")
            return False

        execute_joint_path(
            robot_model, robot_data, [q_sweep], ik_renderer,
            target_pos=sweeping_pos, goal_pos=goal_pos, use_direct_torque=True, render_stride=10,
            gripper_ctrl=gripper_closed, hold_steps=100, wait_on_exit=False,
            reset_start_pose=False,
            overlay_fn=overlay_fn,
        )

        sweep_targets = [d for d in detected_objects if d.get("class") != "broom"]
        q_curr = q_sweep
        current_pos = sweeping_pos.copy()
        if not sweep_targets:
            print("No sweepable objects detected; skipping sweep plan.")
        else:
            sweep_targets = sorted(
                sweep_targets,
                key=lambda d: float(d.get("confidence", 0.0)),
                reverse=True,
            )
            for idx, det in enumerate(sweep_targets, 1):
                lift_height = float(lift_pos[2])
                if abs(current_pos[2] - lift_height) > 1e-4:
                    lift_in_place = current_pos.copy()
                    lift_in_place[2] = lift_height
                    print("Lifting in place to clear sweep height:", lift_in_place)
                    q_lift_in_place, ok_lift_in_place = _move_to_target(
                        robot_model, robot_data, lift_in_place, target_rot, q_curr, ik_renderer,
                        gripper_closed, steps=20, reset_start_pose=False, goal_pos=goal_pos, overlay_fn=overlay_fn
                    )
                    if not ok_lift_in_place:
                        print("IK failed for in-place lift; skipping target.")
                        continue
                    q_curr = q_lift_in_place
                    current_pos = lift_in_place
                obj_cam = det.get("position_camera_frame")
                if obj_cam is None:
                    continue
                obj_world = _camera_to_world(robot_model, "overhead_cam", obj_cam.astype(np.float32))
                obj_xy = obj_world[:2]
                vec_xy = obj_xy - goal_xy
                norm_xy = float(np.linalg.norm(vec_xy))
                if norm_xy < 1e-6:
                    vec_xy = np.array([1.0, 0.0], dtype=np.float32)
                    norm_xy = 1.0
                dir_xy = vec_xy / norm_xy

                behind_offset = 0.12
                behind_xy = obj_xy + dir_xy * behind_offset
                lift_z = float(lift_pos[2])
                behind_lift_pos = np.array([behind_xy[0], behind_xy[1], lift_z], dtype=np.float32)
                print(f"Sweeping target {idx}/{len(sweep_targets)} at:", behind_lift_pos)
                _show_marker(ik_renderer, robot_data, behind_lift_pos)
                q_behind, ok_behind = _move_to_target(
                    robot_model, robot_data, behind_lift_pos, target_rot, q_curr, ik_renderer,
                    gripper_closed, steps=35, reset_start_pose=False, marker_pos=behind_lift_pos, goal_pos=goal_pos, overlay_fn=overlay_fn
                )
                if not ok_behind:
                    print("IK failed for lifted behind position; skipping target.")
                    continue
                q_curr = q_behind
                current_pos = behind_lift_pos
                sweep_start_pos = behind_lift_pos.copy()
                sweep_start_pos[2] = sweeping_pos[2]
                print("Lowering to sweep height:", sweep_start_pos)
                q_sweep_start, ok_sweep_start = _move_to_target(
                    robot_model, robot_data, sweep_start_pos, target_rot, q_curr, ik_renderer,
                    gripper_closed, steps=25, reset_start_pose=False, marker_pos=behind_lift_pos, goal_pos=goal_pos, overlay_fn=overlay_fn
                )
                if not ok_sweep_start:
                    print("IK failed for sweep start; skipping target.")
                    continue
                q_curr = q_sweep_start
                current_pos = sweep_start_pos
                sweep_goal_pos = np.array([goal_xy[0], goal_xy[1], sweeping_pos[2]], dtype=np.float32)
                print("Sweeping toward goal:", sweep_goal_pos)
                q_goal_sweep, ok_goal_sweep = _move_to_target(
                    robot_model, robot_data, sweep_goal_pos, target_rot, q_curr, ik_renderer,
                    gripper_closed, steps=45, reset_start_pose=False, marker_pos=behind_lift_pos, goal_pos=goal_pos, overlay_fn=overlay_fn
                )
                if not ok_goal_sweep:
                    print("IK failed for sweep goal; skipping target.")
                    continue
                q_curr = q_goal_sweep
                current_pos = sweep_goal_pos

        count_in_goal = _count_in_goal(robot_model, robot_data, obj_names_present, goal_xy, goal_radius)
        status["success"] = (count_in_goal == len(obj_names_present))
        status["done"] = True

        execute_joint_path(
            robot_model, robot_data, [q_curr], ik_renderer,
            target_pos=sweeping_pos, goal_pos=goal_pos, use_direct_torque=True, render_stride=10,
            gripper_ctrl=gripper_closed, hold_steps=120, wait_on_exit=False,
            reset_start_pose=False,
            overlay_fn=overlay_fn,
        )
        return status["success"]

    print("Warning: no gripper actuators found; cannot close gripper.")
    return False


demo_cases = [
    (
        "Line near gripper",
        lambda model, data, names: _place_objects_line(
            model,
            data,
            names,
            panda_base_xy + np.array([0.06, -0.08], dtype=np.float32),
            np.array([0.0, 1.0], dtype=np.float32),
            spacing=0.06,
        ),
    ),
    ("Spread out", _place_objects_spread),
    ("Cluster far from goal", _place_objects_cluster),
    (
        "Line further from gripper",
        lambda model, data, names: _place_objects_line_far(model, data, names),
    ),
    ("Circle around goal", _place_objects_circle),
]

auto_advance_s = 1.0
for case_name, place_fn in demo_cases:
    print(f"\n=== Demo: {case_name} ===")
    run_demo_case(case_name, place_fn)
    time.sleep(auto_advance_s)



=== Demo: Line near gripper ===
Actuators driving arm: [(0, 'panda_actuator1', 'panda_joint1', 1.0, (-87.0, 87.0)), (1, 'panda_actuator2', 'panda_joint2', 1.0, (-87.0, 87.0)), (2, 'panda_actuator3', 'panda_joint3', 1.0, (-87.0, 87.0)), (3, 'panda_actuator4', 'panda_joint4', 1.0, (-87.0, 87.0)), (4, 'panda_actuator5', 'panda_joint5', 1.0, (-12.0, 12.0)), (5, 'panda_actuator6', 'panda_joint6', 1.0, (-12.0, 12.0)), (6, 'panda_actuator7', 'panda_joint7', 1.0, (-12.0, 12.0))]
Total actuators: 8
Updated actuator forcerange: [(np.float64(-200.0), np.float64(200.0)), (np.float64(-200.0), np.float64(200.0)), (np.float64(-200.0), np.float64(200.0)), (np.float64(-200.0), np.float64(200.0)), (np.float64(-80.0), np.float64(80.0)), (np.float64(-80.0), np.float64(80.0)), (np.float64(-80.0), np.float64(80.0))]
Arm dof adr: [np.int32(30), np.int32(31), np.int32(32), np.int32(33), np.int32(34), np.int32(35), np.int32(36)]
Hold max |tau|: 22.2513831852774
Actuators driving arm: [(0, 'panda_actuator1', '